# Baselines

## Environment preparation

In [ ]:
import os
import torch
from functools import partial

from services.index import Index, IndexDataset
from services.baseline.data_process_utils import process_elements_main
from services.common.calibration_heads import BetaCalibrationHead, TemperatureCalibrationHead
from services.baseline.calibration_utils import (
    fit_hparameters_beta,
    fit_hparameters_temp,
    test_calibration_model
)

In [17]:
HPARAMETER_SEARCH_TRIALS = 20
FEATURES_COUNT = 0

LOGS_DIR = "./logs/"

In [ ]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")

CUDA_VISIBLE_DEVICES=3


In [19]:
torch.random.manual_seed(42);

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"Index {i}: {props.name}, UUID: {props.uuid}")
    
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(device)
print(f"Chosen device: {device}")

Index 0: NVIDIA L40, UUID: 7b7e27a5-a434-d736-eabd-b9414645af76
Chosen device: cuda:0


## Data preparation 

In [20]:
index = Index("../../../../index_data/llama3_RACE_attn_cropped_12000")

## Splits preparing

In [21]:
train = IndexDataset(
    index, 
    partial(
        process_elements_main,
        device=device,
    ),
    split="train",
    load_all_data=True,
    verbose=True
)
val = IndexDataset(
    index, 
    partial(
        process_elements_main,
        device=device,
    ),
    split="val",
    load_all_data=True,
    verbose=True
)
test = IndexDataset(
    index, 
    partial(
        process_elements_main,
        device=device,
    ),
    split="test",
    load_all_data=True,
    verbose=True
)

Processing data...: 100%|██████████| 1199/1199 [00:00<00:00, 3944.06it/s]


### No calibration

In [22]:
logs_dir = LOGS_DIR + "(raw)calibration_res/"

test_data = test.get()
raw_test_probs = test_data["features"]
if raw_test_probs.ndim > 1:
    raw_test_probs = raw_test_probs.squeeze(-1)

test_calibration_model(
    raw_test_probs,
    test_data["labels"],
    device=device,
    logging=True,
    log_dir=logs_dir + "test",
);

### Beta calibration

In [23]:
logs_dir = LOGS_DIR + "(beta)calibration_res/"

test_data = test.get()
fit_results = fit_hparameters_beta(
    model_class=BetaCalibrationHead,
    train=train,
    test=val,
    features_count=FEATURES_COUNT,
    device=device,
    logging=True,
    log_dir=logs_dir + f"train"
)

model = BetaCalibrationHead(
    in_features=FEATURES_COUNT + 1,
    device=device
)
model.load_state_dict(fit_results["parameters"])
model.eval()

test_calibrated_probs = model.calibrate(test_data["features"], device)
test_calibration_model(
    test_calibrated_probs,
    test_data["labels"],
    device=device,
    logging=True,
    log_dir=logs_dir + f"test",
);

### Temperature calibration

In [24]:
logs_dir = LOGS_DIR + "(temperature)calibration_res/"

test_data = test.get()
fit_results = fit_hparameters_temp(
    model_class=TemperatureCalibrationHead,
    train=train,
    test=val,
    features_count=FEATURES_COUNT,
    device=device,
    logging=True,
    log_dir=logs_dir + f"train"
)

model = TemperatureCalibrationHead(
    in_features=FEATURES_COUNT + 1,
    device=device
)
model.load_state_dict(fit_results["parameters"])
model.eval()

test_calibrated_probs = model.calibrate(test_data["logits"], device)
test_calibrated_probs = test_calibrated_probs.gather(
    1, test_data["gen_tok_ids"].unsqueeze(1)
).squeeze(1)
test_calibration_model(
    test_calibrated_probs,
    test_data["labels"],
    device=device,
    logging=True,
    log_dir=logs_dir + f"test",
);